# 00. Colab 환경 및 데이터 준비

저장소를 고정된 위치에 준비하고, 의존성·데이터·테스트·fold를 검증한다.

- 입력: 추적된 `dataset/*.jsonl`, 기존 zero-shot 결과
- 출력: `artifacts/reports/*`, `artifacts/folds/*`
- 다음 단계: `01_zero_shot_baseline.ipynb` 또는 `02_cpu_baselines.ipynb`

긴 학습을 이어 할 경우 모든 세션에서 같은 `PROJECT_ROOT`, `artifacts` Drive 경로,
`HF_HOME`을 사용해야 한다. 이미 생성된 artifact를 다른 경로로 옮기지 않는다.


## 1. 저장소와 영속 경로


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


## 2. CPU/테스트 의존성 설치


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 3. 입력 및 런타임 사전 점검


In [ ]:
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
ZERO_SHOT_PATH = (
    PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"
    / "qwen3_14b_zero_shot_strict_v2_validation.jsonl"
)
require_paths(TRAIN_PATH, VALIDATION_PATH, ZERO_SHOT_PATH)

def count_jsonl(path: Path) -> int:
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())

total, used, free = shutil.disk_usage(PROJECT_ROOT)
print({
    "python": sys.version.split()[0],
    "train_rows": count_jsonl(TRAIN_PATH),
    "validation_rows": count_jsonl(VALIDATION_PATH),
    "disk_free_gb": round(free / 1024**3, 1),
})
assert (3, 10) <= sys.version_info[:2] < (3, 13)


## 4. 실행 스위치

테스트만 기본 실행한다. 감사와 fold CLI는 산출물을 덮어쓰지 않으므로, 처음 실행할
때만 스위치를 켠다. 이미 결과가 있으면 삭제하지 말고 그대로 재사용한다.


In [ ]:
RUN_UNIT_TESTS = True
RUN_DATA_AUDIT = False
RUN_ZERO_SHOT_REAUDIT = False
RUN_MAIN_FOLDS = False
RUN_LOPO_FOLDS = False


In [ ]:
run_cli("unit tests", RUN_UNIT_TESTS, "-m", "pytest", "-q")


## 5. 데이터 및 기존 zero-shot 결과 감사


In [ ]:
run_cli(
    "data audit",
    RUN_DATA_AUDIT,
    "scripts/audit_data.py",
    "--config", "configs/data.yaml",
)
run_cli(
    "zero-shot re-audit",
    RUN_ZERO_SHOT_REAUDIT,
    "scripts/reproduce_zero_shot.py",
    "--config", "configs/data.yaml",
)

show_json(PROJECT_ROOT / "artifacts" / "reports" / "data_audit.json")
show_json(PROJECT_ROOT / "artifacts" / "reports" / "zero_shot_reaudit.json")


## 6. 고정 5-fold 및 LOPO 진단 fold


In [ ]:
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
LOPO_PATH = PROJECT_ROOT / "artifacts" / "folds" / "lopo_by_prompt.jsonl"

run_cli(
    "main 5-fold",
    RUN_MAIN_FOLDS,
    "scripts/make_folds.py",
    "--config", "configs/data.yaml",
    "--n-folds", "5",
    "--seed", "42",
    "--output", FOLDS_PATH,
)
run_cli(
    "LOPO diagnostic folds",
    RUN_LOPO_FOLDS,
    "scripts/make_lopo_folds.py",
    "--config", "configs/data.yaml",
    "--output", LOPO_PATH,
)

for path in (FOLDS_PATH, LOPO_PATH):
    print(path, "ready" if path.exists() else "not created")


## 7. 다음 단계

- 신규 zero-shot 생성이 필요하면 `01_zero_shot_baseline.ipynb`
- CPU OOF 기준선은 `02_cpu_baselines.ipynb`
- validation label을 합친 최종 재훈련은 일반 개발 흐름과 분리된
  `90_optional_final_retraining.ipynb`에서만 수행한다.
